<a href="https://colab.research.google.com/github/pradeepkr12/kaggle/blob/main/TorchTPU_0_2_0_Gemma2B_Inference_Optimizations_v5e_F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TorchTPU · Gemma 2B greedy decoding on TPU v5e-1: Boosting inference to 21,645 tok/s


**Author:** Rubens Zimbres, Google Developer Expert

**Based on the article:** *From 1,540 to 19,511 Tokens per Second on a Single TPU v5e Chip: A Measurement Study of Large Language Model Inference Optimization*, Rubens de Almeida Zimbres, July 2026.

**Original article:** https://zenodo.org/records/21221952

**License:** Copyright 2026 Rubens Zimbres. This notebook is licensed under the Apache License, Version 2.0. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0. The original article by Rubens de Almeida Zimbres is published separately under its own terms at https://zenodo.org/records/23840458.

This notebook re-runs the study (*From 1,540 to 19,511 Tokens per Second on a Single TPU v5e Chip: A Measurement Study of Large Language Model Inference Optimization* at: [Zenodo article](https://zenodo.org/records/21221952) ) on the **public `torch_tpu==0.2.0`** release (pinned: torch 2.11.0+cpu, jax/jaxlib 0.9.2, libtpu 0.0.50) on the **same Colab v5e-1 runtime** as the vLLM notebook (19,511 tok/s, batch 512, BKV=256). It ports every applicable optimization from that notebook and adds the levers that are specific to TorchTPU.

| JAX + vLLM notebook | Here (TorchTPU 0.2.0) |
|---|---|
| `temperature=0.0` (greedy) | `argmax` on device inside the compiled step. No sampler, RNG, or per-token host transfer; tokens are copied to the host once at the end |
| RPA v3 kernel vendored + `RPA_BKV` monkeypatch | Our own flash-decoding Pallas kernel registered with `torch.tpu.pallas.jax_op` (no patching). Same `RPA_BKV` knob, plus **block skipping** past `pos` and a batch-block knob `PALLAS_BB` |
| `max_num_seqs=512`, `BATCH_MULT=128` | Batch 512 for the kernel, scan and headline experiments |
| Batch TTFT, ITL, energy ceiling, metrics block | The same formulas and the same block, plus `[EVIDENCE]` lines and a `Quotable` flag |
| ngram speculative decoding | **Skipped**: it was OFF in the 19,511 run, needs real weights to accept tokens, and variable acceptance forces host syncs |

**What changed since my starting study (wheel 0.1.1.dev20260713 → 0.2.0), so these must be re-measured:**
* The default eager mode is now **strict** (`DEFER_NEVER`). Fuse is opt-in. The draft's "as shipped = fused" regime is no longer the default.
* The upstream Glimmer-30B case study shows torch.compile + StaticCache at ~23.6 ms/step. The ~430 ms compiled-call overhead may be gone.
* `torch.ops.higher_order.scan` lowers to `create_scan_op` under torch.compile. That makes the "CUDA Graphs analogue" from §8.1 testable (`DECODE_SCAN=K`).

**Rules (unchanged from the study):** one variable per run; every run is a separate process (libtpu allows one process per chip, so *this notebook kernel never touches the TPU*); profiled, step-timed, and debug runs are marked non-quotable and the analysis drops them; CPU fallback off; gradients off; and every treatment prints evidence that it is active.

**Four experiments, 9 runs:** (1) eager vs torch.compile at batch 128; (2) XLA vs Pallas kernel at `RPA_BKV` 64/128/256, batch 512; (3) multi-step `scan` decode, K = 8 and 32; (4) real weights at batch 512 with the best configuration. Each run costs roughly 1 to 3 minutes, mostly compilation, so about 20 to 30 minutes in total.

## 0. Setup
Install the pinned 0.2.0 stack. **Restart the session after this cell** (Runtime → Restart session) so the new torch is the one that gets imported.

In [ ]:
!pip uninstall -y -q torchvision torchaudio
!pip install -q torch_tpu==0.2.0 "torch==2.11.0" "jax==0.9.2" "jaxlib==0.9.2" "libtpu==0.0.50" \
    --extra-index-url https://download.pytorch.org/whl/cpu
!pip install -q huggingface_hub safetensors "transformers>=4.45,<5" sentencepiece \
    tensorboard tensorboard-plugin-profile pandas
!pip list 2>/dev/null | grep -iE "^(torch|torch[-_]tpu|jax|jaxlib|libtpu|transformers) "

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.12 requires torchvision>=0.11, which is not installed.
jax                          0.9.2
jaxlib                       0.9.2
libtpu                       0.0.50
torch                        2.11.0+cpu
torch_tpu                    0.2.0
transformers                 4.57.6


In [ ]:
!mkdir -p /content/torchtpu_project /content/torchtpu_logs /content/torchtpu_cache /content/torchtpu_profile

### Project files
* `gemma2b.py`: a hand-written Gemma 1 2B with HF attribute names, a static KV cache, a 0-D device position, and optional fused QKV / gate-up.
* `decode_kernel.py`: the flash-decoding Pallas kernel (pure JAX, so it can be checked on CPU).
* `attention_ops.py`: registers that kernel with `torch.tpu.pallas.jax_op`, with a trace-time probe.
* `run_torchtpu.py`: the harness, configured by environment variables (Appendix A, extended).
* `verify_*_cpu.py`: CPU correctness checks.

In [ ]:
%%writefile /content/torchtpu_project/gemma2b.py
"""Gemma 1 (2B) in plain PyTorch, written for static-shape decoding on TorchTPU.

MQA: 8 query heads share 1 KV head, head_dim 256, 18 layers, hidden 2048,
GELU (tanh), RMSNorm in float32 with (1 + w), tied embeddings scaled by
sqrt(hidden). Attribute names match the Hugging Face checkpoint so
google/gemma-2b loads directly (q/k/v and gate/up are optionally fused at
load time).

Static-shape rules (see the TorchTPU execution-model docs and the Glimmer
StaticCache example):
  * The KV cache is preallocated, shape (B, MAX_LEN, D) per layer.
  * The decode position is a 0-D device tensor, never a Python int, so
    torch.compile(dynamic=False) never guards on it.
  * Masks and RoPE are computed in-graph from that tensor.
  * Cache writes use index_copy_ (in place) or index_copy (functional).
"""
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

NEG = -1e30


@dataclass(frozen=True)
class GemmaConfig:
  vocab_size: int = 256000
  hidden: int = 2048
  intermediate: int = 16384
  n_layers: int = 18
  n_heads: int = 8
  head_dim: int = 256
  eps: float = 1e-6
  rope_theta: float = 10000.0


def rotate_half(x):
  x1, x2 = x.chunk(2, dim=-1)
  return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
  xf = x.float()
  return (xf * cos + rotate_half(xf) * sin).to(x.dtype)


class GemmaRMSNorm(nn.Module):

  def __init__(self, dim, eps):
    super().__init__()
    self.eps = eps
    self.weight = nn.Parameter(torch.zeros(dim))

  def forward(self, x):
    xf = x.float()
    xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
    return (xf * (1.0 + self.weight.float())).to(x.dtype)


class GemmaAttention(nn.Module):

  def __init__(self, cfg, fused):
    super().__init__()
    self.h, self.d, self.fused = cfg.n_heads, cfg.head_dim, fused
    if fused:
      self.qkv_proj = nn.Linear(cfg.hidden, (cfg.n_heads + 2) * cfg.head_dim,
                                bias=False)
    else:
      self.q_proj = nn.Linear(cfg.hidden, cfg.n_heads * cfg.head_dim, bias=False)
      self.k_proj = nn.Linear(cfg.hidden, cfg.head_dim, bias=False)
      self.v_proj = nn.Linear(cfg.hidden, cfg.head_dim, bias=False)
    self.o_proj = nn.Linear(cfg.n_heads * cfg.head_dim, cfg.hidden, bias=False)

  def forward(self, x, cos, sin, kc, vc, widx, attend, inplace, use_cache):
    B, T, _ = x.shape
    if self.fused:
      # Zero-copy split of one projection (FLUX case-study milestone 1).
      q, k, v = self.qkv_proj(x).split([self.h * self.d, self.d, self.d], -1)
    else:
      q, k, v = self.q_proj(x), self.k_proj(x), self.v_proj(x)
    q = apply_rope(q.view(B, T, self.h, self.d), cos[None, :, None], sin[None, :, None])
    k = apply_rope(k, cos[None], sin[None])          # (B, T, D): the single KV head
    if inplace:
      kc.index_copy_(1, widx, k)
      vc.index_copy_(1, widx, v)
    else:
      kc = kc.index_copy(1, widx, k)
      vc = vc.index_copy(1, widx, v)
    o = attend(q, kc, vc) if use_cache else attend(q, k, v)
    return self.o_proj(o.reshape(B, T, self.h * self.d)), kc, vc


class GemmaMLP(nn.Module):

  def __init__(self, cfg, fused):
    super().__init__()
    self.fused = fused
    if fused:
      self.gate_up_proj = nn.Linear(cfg.hidden, 2 * cfg.intermediate, bias=False)
    else:
      self.gate_proj = nn.Linear(cfg.hidden, cfg.intermediate, bias=False)
      self.up_proj = nn.Linear(cfg.hidden, cfg.intermediate, bias=False)
    self.down_proj = nn.Linear(cfg.intermediate, cfg.hidden, bias=False)

  def forward(self, x):
    if self.fused:
      g, u = self.gate_up_proj(x).chunk(2, dim=-1)
    else:
      g, u = self.gate_proj(x), self.up_proj(x)
    return self.down_proj(F.gelu(g, approximate="tanh") * u)


class GemmaDecoderLayer(nn.Module):

  def __init__(self, cfg, fused):
    super().__init__()
    self.self_attn = GemmaAttention(cfg, fused)
    self.mlp = GemmaMLP(cfg, fused)
    self.input_layernorm = GemmaRMSNorm(cfg.hidden, cfg.eps)
    self.post_attention_layernorm = GemmaRMSNorm(cfg.hidden, cfg.eps)

  def forward(self, x, cos, sin, kc, vc, widx, attend, inplace, use_cache):
    h, kc, vc = self.self_attn(self.input_layernorm(x), cos, sin, kc, vc, widx,
                               attend, inplace, use_cache)
    x = x + h
    x = x + self.mlp(self.post_attention_layernorm(x))
    return x, kc, vc


class GemmaModel(nn.Module):

  def __init__(self, cfg, fused):
    super().__init__()
    self.embed_tokens = nn.Embedding(cfg.vocab_size, cfg.hidden)
    self.layers = nn.ModuleList(
        [GemmaDecoderLayer(cfg, fused) for _ in range(cfg.n_layers)])
    self.norm = GemmaRMSNorm(cfg.hidden, cfg.eps)


# ---------------------------------------------------------------- attention
def prefill_attention(q, k, v, bias):
  """q (B,P,H,D), k/v (B,P,D), bias (B,P,P) f32 -> (B,P,H,D)."""
  s = torch.einsum("bqhd,bkd->bhqk", q, k).float() * q.shape[-1] ** -0.5
  p = torch.softmax(s + bias[:, None], dim=-1).to(q.dtype)
  return torch.einsum("bhqk,bkd->bqhd", p, v)


def decode_attention_xla(q, kc, vc, bias):
  """q (B,1,H,D), kc/vc (B,L,D), bias (B,L) f32 -> (B,1,H,D). MQA as one bmm."""
  s = torch.bmm(q[:, 0], kc.transpose(1, 2)).float() * q.shape[-1] ** -0.5
  p = torch.softmax(s + bias[:, None, :], dim=-1).to(q.dtype)
  return torch.bmm(p, vc)[:, None]


def _bias(valid):
  return torch.full(valid.shape, NEG, dtype=torch.float32,
                    device=valid.device).masked_fill(valid, 0.0)


class Gemma(nn.Module):

  def __init__(self, cfg=GemmaConfig(), fused=True):
    super().__init__()
    self.cfg, self.fused = cfg, fused
    self.model = GemmaModel(cfg, fused)
    # HF Gemma casts the normalizer to the activation dtype: sqrt(2048) -> 45.25.
    self.normalizer = float(torch.tensor(cfg.hidden ** 0.5, dtype=torch.bfloat16,
                                         device="cpu"))
    self.register_buffer("inv_freq", self._inv_freq(), persistent=False)

  def _inv_freq(self, device=None):
    d = self.cfg.head_dim
    return 1.0 / (self.cfg.rope_theta ** (
        torch.arange(0, d, 2, dtype=torch.float32, device=device) / d))

  def reset_buffers(self, device):
    self.inv_freq = self._inv_freq(device)

  def embed(self, tokens):
    return self.model.embed_tokens(tokens) * self.normalizer

  def rope(self, positions):
    freqs = positions[:, None] * self.inv_freq[None, :]
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos(), emb.sin()

  def logits(self, x):
    return F.linear(self.model.norm(x), self.model.embed_tokens.weight)

  def _run_layers(self, x, cos, sin, ks, vs, widx, attend, inplace, use_cache):
    nks, nvs = [], []
    for i, layer in enumerate(self.model.layers):
      x, kc, vc = layer(x, cos, sin, ks[i], vs[i], widx, attend, inplace, use_cache)
      nks.append(kc)
      nvs.append(vc)
    return x, nks, nvs

  def prefill(self, tokens, start, ks, vs, *, inplace=True):
    """Left-padded prompt batch (B,P) -> greedy next token (B,), caches filled 0..P-1."""
    _, P = tokens.shape
    idx = torch.arange(P, device=tokens.device)
    cos, sin = self.rope(idx.to(torch.float32))
    key = idx[None, None, :]
    valid = (key >= start[:, None, None]) & (key <= idx[None, :, None])
    bias = _bias(valid)
    x, nks, nvs = self._run_layers(
        self.embed(tokens), cos, sin, ks, vs, idx,
        lambda q, k, v: prefill_attention(q, k, v, bias), inplace, False)
    return torch.argmax(self.logits(x[:, -1]), dim=-1), nks, nvs

  def decode(self, cur, pos, start, ks, vs, *, attn="xla", inplace=True,
             bkv=128, bb=8):
    """One greedy step: token at position `pos` (0-D tensor) -> next token (B,)."""
    L = ks[0].shape[1]
    cos, sin = self.rope(pos.to(torch.float32).view(1))
    if attn == "pallas":
      from attention_ops import pallas_decode_attention
      start32, pos32 = start.to(torch.int32), pos.to(torch.int32).view(1)
      attend = lambda q, kc, vc: pallas_decode_attention(
          q[:, 0], kc, vc, start32, pos32, bkv, bb)[:, None]
    else:
      kv = torch.arange(L, device=cur.device)[None, :]
      bias = _bias((kv >= start[:, None]) & (kv <= pos))
      attend = lambda q, kc, vc: decode_attention_xla(q, kc, vc, bias)
    x, nks, nvs = self._run_layers(
        self.embed(cur[:, None]), cos, sin, ks, vs, pos.view(1), attend,
        inplace, True)
    # Greedy, temperature 0: argmax on device, no sampler, no host round trip.
    return torch.argmax(self.logits(x[:, 0]), dim=-1), nks, nvs

  def forward_full(self, tokens, start):
    """Uncached reference forward over (B,T) -> logits (B,T,V). CPU checks only."""
    _, T = tokens.shape
    idx = torch.arange(T, device=tokens.device)
    cos, sin = self.rope(idx.to(torch.float32))
    key = idx[None, None, :]
    bias = _bias((key >= start[:, None, None]) & (key <= idx[None, :, None]))
    x = self.embed(tokens)
    for layer in self.model.layers:
      h = layer.input_layernorm(x)
      a = layer.self_attn
      B = h.shape[0]
      if a.fused:
        q, k, v = a.qkv_proj(h).split([a.h * a.d, a.d, a.d], -1)
      else:
        q, k, v = a.q_proj(h), a.k_proj(h), a.v_proj(h)
      q = apply_rope(q.view(B, T, a.h, a.d), cos[None, :, None], sin[None, :, None])
      k = apply_rope(k, cos[None], sin[None])
      o = prefill_attention(q, k, v, bias)
      x = x + a.o_proj(o.reshape(B, T, a.h * a.d))
      x = x + layer.mlp(layer.post_attention_layernorm(x))
    return self.logits(x)


@torch.no_grad()
def init_random_(model, std=0.02, seed=0):
  """Random weights with real shapes: valid for throughput, not for text."""
  torch.manual_seed(seed)
  for name, p in model.named_parameters():
    if "norm" in name:
      p.zero_()            # Gemma RMSNorm scales by (1 + w)
    else:
      p.normal_(0.0, std)  # generated on the device, no host copy


@torch.no_grad()
def load_hf_state_(model, state):
  """Copies a google/gemma-2b state dict into `model`, asserting full coverage."""
  used = set()

  def take(k):
    used.add(k)
    return state[k]

  def put(param, t):
    param.copy_(t.to(param.dtype))

  m = model.model
  put(m.embed_tokens.weight, take("model.embed_tokens.weight"))
  put(m.norm.weight, take("model.norm.weight"))
  for i, layer in enumerate(m.layers):
    p = f"model.layers.{i}."
    a, mlp = layer.self_attn, layer.mlp
    q, k, v = (take(p + f"self_attn.{n}_proj.weight") for n in "qkv")
    if a.fused:
      put(a.qkv_proj.weight, torch.cat([q, k, v], 0))
    else:
      put(a.q_proj.weight, q), put(a.k_proj.weight, k), put(a.v_proj.weight, v)
    put(a.o_proj.weight, take(p + "self_attn.o_proj.weight"))
    g, u = take(p + "mlp.gate_proj.weight"), take(p + "mlp.up_proj.weight")
    if mlp.fused:
      put(mlp.gate_up_proj.weight, torch.cat([g, u], 0))
    else:
      put(mlp.gate_proj.weight, g), put(mlp.up_proj.weight, u)
    put(mlp.down_proj.weight, take(p + "mlp.down_proj.weight"))
    put(layer.input_layernorm.weight, take(p + "input_layernorm.weight"))
    put(layer.post_attention_layernorm.weight,
        take(p + "post_attention_layernorm.weight"))
  unused = sorted(set(state) - used)
  assert not unused, f"checkpoint tensors not consumed: {unused[:5]}"
  return len(used)

Writing /content/torchtpu_project/gemma2b.py


In [ ]:
%%writefile /content/torchtpu_project/decode_kernel.py
"""Flash-decoding Pallas kernel for Gemma 2B MQA decode (one query token per sequence).

Pure JAX: importable without torch_tpu, so it can be verified on CPU in
interpret mode before it ever touches the TPU.

Layout (the single KV head of Gemma 2B is squeezed away):
  q:     (B, H, D)   H = 8 query heads, D = 256
  k, v:  (B, L, D)   dense KV cache, L = MAX_LEN
  start: (B,) int32  first valid cache slot per sequence (left padding)
  pos:   (1,) int32  current position; slots > pos are not yet written

Grid = (B // bb, L // bkv). The KV axis is the online-softmax reduction
("arbitrary"); the batch axis is "parallel". Per-sequence scalars travel as
whole arrays in SMEM through scalar prefetch and are indexed by the grid id.

The kernel hack (the TorchTPU analogue of the RPA_BKV override in the vLLM
study): bkv is the KV block size, and KV blocks that lie entirely past `pos`
are skipped. Their index_map is clamped to the last live block, so Pallas
does not issue a new DMA for them and the body does no work.
"""
import functools

import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu

NEG = -1e30
_LANES = 128
_CompilerParams = getattr(pltpu, "CompilerParams", None) or getattr(
    pltpu, "TPUCompilerParams")


def _kernel(pos_ref, start_ref, q_ref, k_ref, v_ref, o_ref, m_sc, l_sc, acc_sc,
            *, bb, bkv, n_heads, scale):
  b = pl.program_id(0)
  j = pl.program_id(1)
  pos = pos_ref[0]

  @pl.when(j == 0)
  def _init():
    m_sc[...] = jnp.full(m_sc.shape, NEG, jnp.float32)
    l_sc[...] = jnp.zeros(l_sc.shape, jnp.float32)
    acc_sc[...] = jnp.zeros(acc_sc.shape, jnp.float32)

  @pl.when(j * bkv <= pos)
  def _body():
    kv_idx = j * bkv + jax.lax.broadcasted_iota(jnp.int32, (n_heads, bkv), 1)
    for i in range(bb):
      rows = pl.ds(i * n_heads, n_heads)
      start = start_ref[b * bb + i]
      q = q_ref[i]                                   # (H, D)
      k = k_ref[i]                                   # (bkv, D)
      v = v_ref[i]                                   # (bkv, D)
      s = jax.lax.dot_general(
          q, k, (((1,), (1,)), ((), ())),
          preferred_element_type=jnp.float32) * scale  # (H, bkv)
      valid = (kv_idx >= start) & (kv_idx <= pos)
      s = jnp.where(valid, s, NEG)
      # Statistics are stored broadcast across 128 lanes; all lanes are equal.
      m_prev = jnp.max(m_sc[rows, :], axis=-1, keepdims=True)
      l_prev = jnp.max(l_sc[rows, :], axis=-1, keepdims=True)
      m_new = jnp.maximum(m_prev, jnp.max(s, axis=-1, keepdims=True))
      p = jnp.where(valid, jnp.exp(s - m_new), 0.0)
      alpha = jnp.exp(m_prev - m_new)
      l_new = alpha * l_prev + jnp.sum(p, axis=-1, keepdims=True)
      pv = jax.lax.dot_general(
          p.astype(v.dtype), v, (((1,), (0,)), ((), ())),
          preferred_element_type=jnp.float32)          # (H, D)
      acc_sc[rows, :] = alpha * acc_sc[rows, :] + pv
      m_sc[rows, :] = jnp.broadcast_to(m_new, (n_heads, _LANES))
      l_sc[rows, :] = jnp.broadcast_to(l_new, (n_heads, _LANES))

  @pl.when(j == pl.num_programs(1) - 1)
  def _finish():
    for i in range(bb):
      rows = pl.ds(i * n_heads, n_heads)
      l = jnp.max(l_sc[rows, :], axis=-1, keepdims=True)
      o_ref[i] = (acc_sc[rows, :] / l).astype(o_ref.dtype)


def decode_attention(q, k, v, start, pos, *, bkv=128, bb=8, interpret=False):
  """Returns softmax(q k^T * D^-0.5 + mask) v with shape (B, H, D)."""
  B, H, D = q.shape
  L = k.shape[1]
  bkv = min(bkv, L)
  assert L % bkv == 0, f"MAX_LEN={L} must be a multiple of bkv={bkv}"
  assert B % bb == 0, f"batch={B} must be a multiple of bb={bb}"
  assert H % 8 == 0, "query heads must fill whole sublanes"

  def kv_map(b, j, pos_ref, start_ref):
    return (b, jnp.minimum(j, pos_ref[0] // bkv), 0)

  def q_map(b, j, pos_ref, start_ref):
    return (b, 0, 0)

  grid_spec = pltpu.PrefetchScalarGridSpec(
      num_scalar_prefetch=2,
      grid=(B // bb, L // bkv),
      in_specs=[
          pl.BlockSpec((bb, H, D), q_map),
          pl.BlockSpec((bb, bkv, D), kv_map),
          pl.BlockSpec((bb, bkv, D), kv_map),
      ],
      out_specs=pl.BlockSpec((bb, H, D), q_map),
      scratch_shapes=[
          pltpu.VMEM((bb * H, _LANES), jnp.float32),
          pltpu.VMEM((bb * H, _LANES), jnp.float32),
          pltpu.VMEM((bb * H, D), jnp.float32),
      ],
  )
  kernel = functools.partial(
      _kernel, bb=bb, bkv=bkv, n_heads=H, scale=D ** -0.5)
  return pl.pallas_call(
      kernel,
      grid_spec=grid_spec,
      out_shape=jax.ShapeDtypeStruct(q.shape, q.dtype),
      compiler_params=_CompilerParams(
          dimension_semantics=("parallel", "arbitrary")),
      interpret=interpret,
  )(pos.astype(jnp.int32), start.astype(jnp.int32), q, k, v)

Writing /content/torchtpu_project/decode_kernel.py


In [ ]:
%%writefile /content/torchtpu_project/attention_ops.py
"""Registers the Pallas decode kernel as a first-class PyTorch op.

torch.tpu.pallas.jax_op replaces the vLLM study's runtime monkeypatching:
no vendoring, no module-reference rebinding. Non-tensor arguments (bkv, bb)
are inferred as static, so each block size is its own compiled kernel.
"""
import jax
import torch

from decode_kernel import decode_attention as _jax_decode_attention

_traced = set()


def _decode_attention(q: jax.Array, k: jax.Array, v: jax.Array,
                      start: jax.Array, pos: jax.Array,
                      bkv: int, bb: int) -> jax.Array:
  key = (tuple(q.shape), tuple(k.shape), bkv, bb)
  if key not in _traced:
    # Fires at trace time only: proof the kernel is compiled onto the hot path.
    _traced.add(key)
    print(f"[MY PALLAS KERNEL] traced q={tuple(q.shape)} kv={tuple(k.shape)} "
          f"bkv={min(bkv, k.shape[1])} bb={bb}", flush=True)
  return _jax_decode_attention(q, k, v, start, pos, bkv=bkv, bb=bb)


pallas_decode_attention = torch.tpu.pallas.jax_op(
    "gemma_bench::decode_attention", _decode_attention)

Writing /content/torchtpu_project/attention_ops.py


In [ ]:
%%writefile /content/torchtpu_project/run_torchtpu.py
"""TorchTPU Gemma 2B greedy-decode harness. One configuration per process.

Ports the protocol of the vLLM run_vllm.py: engine init, warmup, batch TTFT
(prefill + 1 token), a 128-token main run, inter-token latency, and an energy
ceiling, then a metrics block that restates its own regime. All configuration
is through environment variables (Appendix A of the study).
"""
import hashlib
import json
import os
import pathlib
import sys
import time

T0 = time.perf_counter()  # script start: anchors all init metrics


def env(name, default):
  return os.environ.get(name, default)


RUN_NAME = env("RUN_NAME", "run")
DEVICE = env("DEVICE", "tpu")            # cpu = dry run of the harness logic only
TINY = env("TINY", "0") == "1"           # tiny config for dry runs
BATCH_MULT = int(env("BATCH_MULT", "8"))
MODEL = env("MODEL", "random")           # random | google/gemma-2b
MAX_LEN = int(env("MAX_LEN", "256"))
PROMPT_PAD = int(env("PROMPT_PAD", "16"))
MAX_TOKENS = int(env("MAX_TOKENS", "128"))
COMPILE = env("COMPILE", "1") == "1"
EXEC_MODE = env("EXEC_MODE", "strict")   # strict | fuse | debug
CPU_FALLBACK = env("CPU_FALLBACK", "0") == "1"
ATTN = env("ATTN", "xla")                # xla | pallas
RPA_BKV = int(env("RPA_BKV", "128"))
PALLAS_BB = int(env("PALLAS_BB", "8"))
SYNC_EVERY = int(env("SYNC_EVERY", "8")) # 0 = only at the end
INPLACE_KV = env("INPLACE_KV", "1") == "1"
FUSED_QKV = env("FUSED_QKV", "1") == "1"
DECODE_SCAN = int(env("DECODE_SCAN", "0"))  # K > 0: K decode steps per executable
STEP_TIMING = env("STEP_TIMING", "0") == "1"
PROFILE = env("PROFILE", "0") == "1"
DETERMINISM = env("DETERMINISM", "0") == "1"
RATED_W = float(env("RATED_W", "197"))   # spec-sheet value; VERIFY before publishing
PROFILE_DIR = env("PROFILE_DIR", "/content/torchtpu_profile")

# Eager-mode env flags must exist before torch_tpu initializes.
if DEVICE == "tpu" and EXEC_MODE == "fuse":
  os.environ["TORCH_TPU_DEFER_AND_FUSE"] = "1"
if DEVICE == "tpu" and EXEC_MODE == "debug":
  os.environ["TORCH_TPU_LAUNCH_BLOCKING"] = "1"

import torch

if DEVICE == "tpu":
  import torch_tpu  # noqa: F401  (registers the "tpu" device if autoload is off)

from gemma2b import Gemma, GemmaConfig, init_random_, load_hf_state_

dev = torch.device(DEVICE)
QUOTABLE = not (PROFILE or STEP_TIMING or EXEC_MODE == "debug" or DEVICE != "tpu")
evidence = {}


def sync():
  if DEVICE == "tpu":
    torch.accelerator.synchronize()


# ------------------------------------------------------------ device + modes
if DEVICE == "tpu":
  modes = {"strict": torch.tpu.EagerMode.DEFER_NEVER,
           "fuse": torch.tpu.EagerMode.DEFER_AND_FUSE,
           "debug": torch.tpu.EagerMode.DEFER_NEVER_AND_LAUNCH_BLOCKING}
  torch.tpu.eager_mode = modes[EXEC_MODE]
  evidence["eager_mode"] = str(torch.tpu.eager_mode)
  try:
    # Internal knob (not in the public API list): forbid silent CPU execution.
    from torch_tpu._internal import execution_mode as _em
    _em.enable_cpu_fallback = CPU_FALLBACK
    evidence["cpu_fallback"] = _em.enable_cpu_fallback
  except Exception as e:  # pylint: disable=broad-except
    evidence["cpu_fallback"] = f"unknown ({type(e).__name__})"
  try:
    props = torch.tpu.get_device_properties(0)
    hbm = (getattr(props, "hbm_bytes_per_chip", None)
           or getattr(props, "total_memory", 0))  # 0.2.0 exposes total_memory
    evidence["device"] = f"{props.name} | HBM {hbm / 2**30:.2f} GiB"
  except Exception as e:  # pylint: disable=broad-except
    evidence["device"] = f"tpu ({type(e).__name__} reading properties)"
  evidence["tier2_cache"] = os.environ.get("TORCH_TPU_TIER2_COMPILATION_CACHE", "default")
  evidence["tier3_cache"] = os.environ.get("TORCH_TPU_TIER3_COMPILATION_CACHE_ROOT", "off")
else:
  evidence["device"] = "cpu DRY RUN (not a TPU measurement)"

torch.set_default_dtype(torch.bfloat16 if DEVICE == "tpu" else torch.float32)
DT = torch.get_default_dtype()

# ------------------------------------------------------------ model
cfg = (GemmaConfig(vocab_size=512, hidden=128, intermediate=256, n_layers=3,
                   n_heads=8, head_dim=256) if TINY else GemmaConfig())
t = time.perf_counter()
with torch.device("meta"):
  model = Gemma(cfg, fused=FUSED_QKV)
model = model.to_empty(device=dev)
model.reset_buffers(dev)
model.eval().requires_grad_(False)

tokenizer = None
hf_path = None
if MODEL != "random" or not TINY:
  try:
    from huggingface_hub import snapshot_download
    pats = ["tokenizer*", "special_tokens_map.json", "config.json"]
    if MODEL != "random":
      pats.append("*.safetensors")
    hf_path = snapshot_download("google/gemma-2b" if MODEL == "random" else MODEL,
                                allow_patterns=pats,
                                token=os.environ.get("HF_TOKEN") or None)
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(hf_path)
  except Exception as e:  # pylint: disable=broad-except
    if MODEL != "random":
      raise
    print(f"[WARN] tokenizer unavailable ({type(e).__name__}); using synthetic ids",
          flush=True)

if MODEL == "random":
  init_random_(model)
  evidence["weights"] = "random (valid throughput, invalid text)"
else:
  from safetensors.torch import load_file
  state = {}
  for f in sorted(pathlib.Path(hf_path).glob("*.safetensors")):
    state.update(load_file(str(f)))
  n = load_hf_state_(model, state)
  del state
  evidence["weights"] = f"{MODEL} ({n} tensors, all consumed)"
sync()
t_build = time.perf_counter() - t

evidence["grad_enabled_in_run"] = None  # filled inside inference_mode below
evidence["any_param_requires_grad"] = any(p.requires_grad for p in model.parameters())
assert not evidence["any_param_requires_grad"]

# ------------------------------------------------------------ prompts
PROMPTS = ["Chirality is", "The birthday of Al Pacino is", "PyTorch is",
           "Brazil celebrate its independence on the days"] * BATCH_MULT
B, P, L = len(PROMPTS), PROMPT_PAD, MAX_LEN
assert P + MAX_TOKENS - 1 <= L, "MAX_LEN too small for PROMPT_PAD + MAX_TOKENS"
assert DECODE_SCAN < MAX_TOKENS - 1, "DECODE_SCAN must be < MAX_TOKENS - 1"
if tokenizer is not None:
  ids = [tokenizer(p, add_special_tokens=True).input_ids for p in PROMPTS]
else:
  ids = [[2] + [sum(map(ord, w)) % (cfg.vocab_size - 10) + 10 for w in p.split()]
         for p in PROMPTS]
assert max(map(len, ids)) <= P, "prompt longer than PROMPT_PAD"
tok_cpu = torch.zeros(B, P, dtype=torch.long)
for i, seq in enumerate(ids):
  tok_cpu[i, P - len(seq):] = torch.tensor(seq)
tokens = tok_cpu.to(dev)
start = torch.tensor([P - len(s) for s in ids], dtype=torch.long, device=dev)

# ------------------------------------------------------------ state + steps
D = cfg.head_dim
# Allocated as inference tensors: compiled-fn outputs are inference tensors too,
# so rebinding functional KV outputs keeps Dynamo's dispatch-key guards stable.
with torch.inference_mode():
  tokens = tokens.clone()
  start = start.clone()
  kv = [[torch.zeros(B, L, D, device=dev) for _ in range(cfg.n_layers)],
        [torch.zeros(B, L, D, device=dev) for _ in range(cfg.n_layers)]]
  pos = torch.zeros((), dtype=torch.long, device=dev)
  cur = torch.zeros(B, dtype=torch.long, device=dev)
  out = torch.zeros(B, MAX_TOKENS, dtype=torch.long, device=dev)
  zero_idx = torch.zeros(1, dtype=torch.long, device=dev)
attn_kw = dict(attn=ATTN, bkv=RPA_BKV, bb=PALLAS_BB)


def prefill_fn(tokens, start, pos, cur, out, zero_idx, ks, vs):
  nxt, nks, nvs = model.prefill(tokens, start, ks, vs, inplace=INPLACE_KV)
  out.index_copy_(1, zero_idx, nxt[:, None])
  cur.copy_(nxt)
  pos.fill_(P)
  return (None, None) if INPLACE_KV else (nks, nvs)


def decode_fn(start, pos, cur, out, ks, vs):
  nxt, nks, nvs = model.decode(cur, pos, start, ks, vs, inplace=INPLACE_KV, **attn_kw)
  out.index_copy_(1, (pos - (P - 1)).view(1), nxt[:, None])
  cur.copy_(nxt)
  pos.add_(1)
  return (None, None) if INPLACE_KV else (nks, nvs)


def scan_fn(start, pos, cur, out, ks, vs):
  """DECODE_SCAN decode steps as one on-device loop (torch.ops.higher_order.scan)."""
  from torch._higher_order_ops import scan as hop_scan
  n = len(ks)

  def body(carry, _x):
    c_cur, c_pos = carry[0], carry[1]
    nxt, nks, nvs = model.decode(c_cur, c_pos, start, list(carry[2:2 + n]),
                                 list(carry[2 + n:]), inplace=False, **attn_kw)
    return [nxt, c_pos + 1, *nks, *nvs], nxt.clone()

  steps = torch.zeros(DECODE_SCAN, device=start.device)
  carry, ys = hop_scan(body, [cur, pos, *ks, *vs], steps)
  gidx = (pos - (P - 1)) + torch.arange(DECODE_SCAN, device=start.device)
  out.index_copy_(1, gidx, ys.transpose(0, 1).contiguous())
  cur.copy_(carry[0])
  pos.copy_(carry[1])
  return list(carry[2:2 + n]), list(carry[2 + n:])


def maybe_compile(fn):
  if not COMPILE:
    return fn
  return torch.compile(fn, backend="tpu" if DEVICE == "tpu" else "aot_eager",
                       dynamic=False, fullgraph=True)


prefill_step = maybe_compile(prefill_fn)
decode_step = maybe_compile(decode_fn)
scan_step = maybe_compile(scan_fn) if DECODE_SCAN > 0 else None


def _rebind(res):
  if res[0] is not None:
    kv[0], kv[1] = list(res[0]), list(res[1])


def generate(n_new, step_times=None):
  _rebind(prefill_step(tokens, start, pos, cur, out, zero_idx, kv[0], kv[1]))
  remaining, done = n_new - 1, 0
  while remaining > 0:
    t_s = time.perf_counter()
    if scan_step is not None and remaining >= DECODE_SCAN:
      _rebind(scan_step(start, pos, cur, out, kv[0], kv[1]))
      k = DECODE_SCAN
    else:
      _rebind(decode_step(start, pos, cur, out, kv[0], kv[1]))
      k = 1
    remaining -= k
    done += k
    if step_times is not None:
      sync()
      step_times.append((time.perf_counter() - t_s) / k)
    elif SYNC_EVERY and done % SYNC_EVERY < k:
      sync()  # bound the async queue depth
  sync()


def graph_count():
  try:
    from torch._dynamo.utils import counters
    return int(counters["stats"]["unique_graphs"])
  except Exception:  # pylint: disable=broad-except
    return -1


# ------------------------------------------------------------ attention check
if ATTN == "pallas" and DEVICE == "tpu":
  from gemma2b import _bias, decode_attention_xla
  from attention_ops import pallas_decode_attention
  with torch.inference_mode():
    q = torch.randn(B, 1, cfg.n_heads, D, device=dev)
    kc, vc = torch.randn(B, L, D, device=dev), torch.randn(B, L, D, device=dev)
    p_chk = torch.tensor(P + 40, device=dev)
    ref = decode_attention_xla(q, kc, vc, _bias(
        (torch.arange(L, device=dev)[None] >= start[:, None])
        & (torch.arange(L, device=dev)[None] <= p_chk)))
    got = pallas_decode_attention(q[:, 0].contiguous(), kc, vc, start.to(torch.int32),
                                  p_chk.to(torch.int32).view(1), RPA_BKV, PALLAS_BB)
    evidence["pallas_vs_xla_max_abs"] = float((got.float() - ref[:, 0].float()).abs().max())

# ------------------------------------------------------------ measurement
prof_ctx = None
with torch.inference_mode():
  evidence["grad_enabled_in_run"] = torch.is_grad_enabled()
  assert not evidence["grad_enabled_in_run"]

  t = time.perf_counter()
  generate(min(MAX_TOKENS, max(8, DECODE_SCAN + 2)))  # compiles prefill, decode (+ scan)
  t_warmup = time.perf_counter() - t
  graphs_after_warmup = graph_count()

  t = time.perf_counter()
  generate(1)                         # batch TTFT proxy: prefill + first token
  t_ttft_batch = time.perf_counter() - t

  if PROFILE:
    from torch.tpu.profiler import TpuProfilerConfig
    pdir = pathlib.Path(PROFILE_DIR) / RUN_NAME
    pdir.mkdir(parents=True, exist_ok=True)
    prof_ctx = torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU,
                    torch.profiler.ProfilerActivity.TPU],
        experimental_config=TpuProfilerConfig(
            host_tracer_level=2, device_tracer_level=1, python_tracer_level=0,
            run_dir=pdir, worker_rank="0"),
        on_trace_ready=torch.profiler.tensorboard_trace_handler(str(pdir)))
    prof_ctx.__enter__()

  step_times = [] if STEP_TIMING else None
  t = time.perf_counter()
  generate(MAX_TOKENS, step_times)    # main run (the only region traced if PROFILE=1)
  out_cpu = out.cpu()                 # results on the host, as vLLM returns them
  t_gen = time.perf_counter() - t
  if prof_ctx is not None:
    prof_ctx.__exit__(None, None, None)

  graphs_after_main = graph_count()
  determinism = None
  if DETERMINISM:
    generate(MAX_TOKENS)
    determinism = bool(torch.equal(out_cpu, out.cpu()))

total_tokens = B * MAX_TOKENS
tps = total_tokens / t_gen
itl_ms = max(t_gen - t_ttft_batch, 1e-9) / (MAX_TOKENS - 1) * 1e3
t_cold = time.perf_counter() - T0
mem = {}
if DEVICE == "tpu":
  try:
    mem["peak_hbm_gib"] = torch.accelerator.max_memory_allocated() / 2**30
  except Exception:  # pylint: disable=broad-except
    pass

regime = (f"{'torch.compile' if COMPILE else 'eager JIT'} | eager_mode={EXEC_MODE} | "
          f"attn={ATTN}{f' bkv={RPA_BKV} bb={PALLAS_BB}' if ATTN == 'pallas' else ''} | "
          f"sync_every={SYNC_EVERY} | inplace_kv={int(INPLACE_KV)} | "
          f"fused_qkv={int(FUSED_QKV)} | decode_scan={DECODE_SCAN}")
print("\n================ METRICS ================")
print(f"Run:                           {RUN_NAME}")
print(f"Regime:                        {regime}")
print(f"Quotable:                      {'YES' if QUOTABLE else 'NO (profiler/step timing/debug/dry run)'}")
for k, v in evidence.items():
  print(f"[EVIDENCE] {k:24s} {v}")
print(f"[EVIDENCE] compiled graphs          after warmup={graphs_after_warmup} "
      f"after main={graphs_after_main} (equal => zero recompiles while measuring)")
print(f"Model build + weights:         {t_build:8.1f} s")
print(f"Warmup generate (compile):     {t_warmup:8.2f} s")
print(f"Engine-init equivalent:        {t_build + t_warmup:8.1f} s  [build + warmup]")
print(f"Batch TTFT (prefill+1 tok):    {t_ttft_batch*1e3:8.0f} ms  [batch={B}]")
print(f"Generation:  {total_tokens} tokens in {t_gen:.2f} s = {tps:,.1f} tok/s")
print(f"Inter-token latency (approx):  {itl_ms:8.1f} ms/token/seq")
if step_times:
  st = sorted(step_times)
  print(f"Per-step device+host time:     median {st[len(st)//2]*1e3:.1f} ms | "
        f"p99 {st[int(len(st)*0.99)]*1e3:.1f} ms  [synced every step: NOT quotable]")
if determinism is not None:
  print(f"Determinism (2 greedy runs):   {'IDENTICAL' if determinism else 'DIFFERENT'}")
if mem:
  print(f"Peak HBM allocated:            {mem['peak_hbm_gib']:8.2f} GiB")
print(f"Greedy tokens checksum:        {hashlib.sha1(out_cpu.numpy().tobytes()).hexdigest()[:12]}")
print(f"Time to first result (cold):   {t_cold:8.1f} s  [script start -> now]")
print(f"Spec-based efficiency:         {tps/RATED_W:8.1f} tok/s per rated W "
      f"(~{RATED_W/tps*1000:.0f} mJ/token upper bound)")
print("==========================================")

if tokenizer is not None and MODEL != "random":
  for i in range(4):
    print(f"\n--- Prompt {i+1}: {PROMPTS[i]}")
    print(f"Generated: {tokenizer.decode(out_cpu[i]).strip()[:200]}")

print("METRICS_JSON " + json.dumps(dict(
    run=RUN_NAME, quotable=QUOTABLE, batch=B, model=MODEL, compile=COMPILE,
    exec_mode=EXEC_MODE, attn=ATTN, bkv=RPA_BKV if ATTN == "pallas" else None,
    bb=PALLAS_BB if ATTN == "pallas" else None, sync_every=SYNC_EVERY,
    inplace_kv=INPLACE_KV, fused_qkv=FUSED_QKV, decode_scan=DECODE_SCAN,
    tok_s=round(tps, 1), itl_ms=round(itl_ms, 2), ttft_ms=round(t_ttft_batch * 1e3, 1),
    build_s=round(t_build, 1), warmup_s=round(t_warmup, 2),
    graphs=graphs_after_main, recompiled=graphs_after_main != graphs_after_warmup,
    step_median_ms=(round(sorted(step_times)[len(step_times)//2] * 1e3, 2)
                    if step_times else None),
    determinism=determinism,
    tokens_sha1=hashlib.sha1(out_cpu.numpy().tobytes()).hexdigest()[:12],
    mj_per_token=round(RATED_W / tps * 1000, 1),
    **{k: v for k, v in evidence.items() if k == "pallas_vs_xla_max_abs"},
    **mem)), flush=True)
sys.exit(0)

Writing /content/torchtpu_project/run_torchtpu.py


In [ ]:
%%writefile /content/torchtpu_project/verify_kernel_cpu.py
"""Numerical verification of the Pallas decode kernel in the CPU interpreter.

Runs with JAX_PLATFORMS=cpu, so it never touches the TPU. Compares against a
float64 NumPy reference across padded, short and full-length windows, for every
block size in the sweep.
"""
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental.pallas import tpu as pltpu

from decode_kernel import decode_attention

INTERPRET = pltpu.InterpretParams() if hasattr(pltpu, "InterpretParams") else True
B, H, D, L = 16, 8, 256, 256
rng = np.random.default_rng(0)


def reference(q, k, v, start, pos):
  q, k, v = (np.asarray(t, np.float64) for t in (q, k, v))
  s = np.einsum("bhd,bld->bhl", q, k) * D ** -0.5
  kv = np.arange(L)[None, None, :]
  valid = (kv >= start[:, None, None]) & (kv <= pos)
  s = np.where(valid, s, -np.inf)
  p = np.exp(s - s.max(-1, keepdims=True))
  p /= p.sum(-1, keepdims=True)
  return np.einsum("bhl,bld->bhd", p, v)


worst = {}
for dtype in (jnp.float32, jnp.bfloat16):
  q = jnp.asarray(rng.standard_normal((B, H, D)), dtype)
  k = jnp.asarray(rng.standard_normal((B, L, D)), dtype)
  v = jnp.asarray(rng.standard_normal((B, L, D)), dtype)
  start = rng.integers(0, 16, B).astype(np.int32)       # left padding up to 15
  for pos in (15, 40, 143, 255):                        # prompt end, short, harness max, full
    ref = reference(q, k, v, start, pos)
    outs = {}
    for bkv in (64, 128, 256):
      out = decode_attention(q, k, v, jnp.asarray(start), jnp.asarray([pos], np.int32),
                             bkv=bkv, bb=8, interpret=INTERPRET)
      outs[bkv] = np.asarray(out, np.float64)
      err = float(np.abs(outs[bkv] - ref).max())
      worst[(dtype.__name__, bkv)] = max(worst.get((dtype.__name__, bkv), 0.0), err)
    spread = max(float(np.abs(outs[a] - outs[b]).max()) for a in outs for b in outs)
    print(f"{dtype.__name__:9s} pos={pos:3d} max|err| per bkv: "
          + " ".join(f"{b}:{np.abs(o - ref).max():.1e}" for b, o in outs.items())
          + f"  | spread across bkv: {spread:.1e}")

print()
for (dt, bkv), e in sorted(worst.items()):
  print(f"[KERNEL VERIFY] {dt:9s} bkv={bkv:3d} worst max|err| vs float64 = {e:.2e}")
assert max(e for (dt, _), e in worst.items() if dt == "float32") < 1e-5, "f32 kernel mismatch"
assert max(e for (dt, _), e in worst.items() if dt == "bfloat16") < 5e-2, "bf16 kernel mismatch"
print("[KERNEL VERIFY] PASS")

Writing /content/torchtpu_project/verify_kernel_cpu.py


In [ ]:
%%writefile /content/torchtpu_project/verify_model_cpu.py
"""CPU correctness checks for gemma2b.py (never touches the TPU).

1. Cached prefill + decode logits == uncached full forward (float32, tiny config).
2. Fused q/k/v + gate/up == unfused, via the checkpoint loader.
3. In-place and functional KV updates produce identical tokens.
"""
import torch

from gemma2b import Gemma, GemmaConfig, init_random_, load_hf_state_

torch.manual_seed(0)
cfg = GemmaConfig(vocab_size=512, hidden=128, intermediate=256, n_layers=3,
                  n_heads=8, head_dim=32)
B, P, L, STEPS = 4, 6, 16, 8

ref = Gemma(cfg, fused=False).float().eval()
init_random_(ref, std=0.2)
with torch.no_grad():
  for n, p in ref.named_parameters():
    if "norm" in n:
      p.normal_(0, 0.1)
state = {k: v.clone() for k, v in ref.state_dict().items() if k != "inv_freq"}
fused = Gemma(cfg, fused=True).float().eval()
n_loaded = load_hf_state_(fused, state)
print(f"[MODEL VERIFY] loader consumed {n_loaded} tensors, fused layout")

lens = torch.tensor([6, 3, 1, 5])
start = P - lens
tokens = torch.randint(1, cfg.vocab_size, (B, P))
tokens[torch.arange(P)[None, :] < start[:, None]] = 0     # left padding


def greedy(model, inplace):
  ks = [torch.zeros(B, L, cfg.head_dim) for _ in range(cfg.n_layers)]
  vs = [torch.zeros(B, L, cfg.head_dim) for _ in range(cfg.n_layers)]
  with torch.inference_mode():
    nxt, ks2, vs2 = model.prefill(tokens, start, ks, vs, inplace=inplace)
    seq = torch.cat([tokens, nxt[:, None]], 1)
    pos = torch.tensor(P)
    for _ in range(STEPS):
      nxt, ks2, vs2 = model.decode(seq[:, -1], pos, start, ks2, vs2, inplace=inplace)
      seq = torch.cat([seq, nxt[:, None]], 1)
      pos = pos + 1
  return seq


with torch.inference_mode():
  seq = greedy(ref, inplace=True)
  # Logits of the cached path vs. the uncached forward over the same sequence.
  full = ref.forward_full(seq[:, :-1], start)
  ks = [torch.zeros(B, L, cfg.head_dim) for _ in range(cfg.n_layers)]
  vs = [torch.zeros(B, L, cfg.head_dim) for _ in range(cfg.n_layers)]
  ref.prefill(seq[:, :P], start, ks, vs)
  err = 0.0
  for t in range(P, seq.shape[1] - 1):
    x = ref.embed(seq[:, t:t + 1])
    cos, sin = ref.rope(torch.tensor([float(t)]))
    kv = torch.arange(L)[None, :]
    from gemma2b import _bias, decode_attention_xla
    bias = _bias((kv >= start[:, None]) & (kv <= t))
    x, _, _ = ref._run_layers(x, cos, sin, ks, vs, torch.tensor([t]),
                              lambda q, kc, vc: decode_attention_xla(q, kc, vc, bias),
                              True, True)
    err = max(err, float((ref.logits(x[:, 0]) - full[:, t]).abs().max()))
print(f"[MODEL VERIFY] cached decode vs uncached forward: max|dlogit| = {err:.2e}")
assert err < 1e-4

seq_f = greedy(fused, inplace=True)
seq_fn = greedy(fused, inplace=False)
assert torch.equal(seq, seq_f), "fused layout changed the greedy tokens"
assert torch.equal(seq_f, seq_fn), "functional KV changed the greedy tokens"
print("[MODEL VERIFY] fused == unfused, in-place == functional (greedy tokens identical)")
print("[MODEL VERIFY] PASS")

Writing /content/torchtpu_project/verify_model_cpu.py


### Runner
Every measurement is a fresh subprocess. `run()` streams only the interesting lines, saves the full log to `/content/torchtpu_logs/<name>.log`, and appends the `METRICS_JSON` line to `results.jsonl`. By default runs share a persistent Tier-3 compilation cache; `cold_cache=True` disables both cache tiers and the AOTAutograd cache.

In [ ]:
import json, os, pathlib, subprocess, sys, tempfile

PROJ = "/content/torchtpu_project"
LOGS = pathlib.Path("/content/torchtpu_logs")
RESULTS = LOGS / "results.jsonl"
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
except Exception as e:
    print("HF_TOKEN not set from Colab secrets:", type(e).__name__)
print("HF_TOKEN length:", len(os.environ.get("HF_TOKEN", "")))

SHOW = ("[EVIDENCE]", "[MY PALLAS", "[KERNEL", "[MODEL", "[WARN]", "Error", "Traceback",
        "Generated:", "--- Prompt")
VERBOSE = False

def run(name, cold_cache=False, timeout=3600, **cfg):
    env = {**os.environ, "PYTHONPATH": PROJ, "PYTHONUNBUFFERED": "1", "RUN_NAME": name,
           "TORCH_TPU_TIER3_COMPILATION_CACHE_ROOT": "/content/torchtpu_cache",
           **{k: str(int(v) if isinstance(v, bool) else v) for k, v in cfg.items()}}
    if cold_cache:
        env.pop("TORCH_TPU_TIER3_COMPILATION_CACHE_ROOT", None)
        env["TORCH_TPU_TIER2_COMPILATION_CACHE"] = "disabled"
        env["TORCHINDUCTOR_CACHE_DIR"] = tempfile.mkdtemp(prefix="cold_inductor_")
    log = LOGS / f"{name}.log"
    print(f"\n>>> {name}  {cfg}{'  [COLD CACHE]' if cold_cache else ''}", flush=True)
    lines, in_block, metrics = [], False, None
    with open(log, "w") as f:
        p = subprocess.Popen([sys.executable, f"{PROJ}/run_torchtpu.py"], env=env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            f.write(line); lines.append(line)
            in_block = in_block or "==== METRICS" in line
            if line.startswith("METRICS_JSON "):
                metrics = json.loads(line[len("METRICS_JSON "):])
            elif VERBOSE or in_block or any(t in line for t in SHOW):
                print(line, end="")
            if in_block and line.startswith("====") and "METRICS" not in line:
                in_block = False
        rc = p.wait(timeout=timeout)
    if metrics is None:
        print(f"!!! {name} FAILED rc={rc} (negative = signal, e.g. -11 segfault). Last lines:")
        print("".join(lines[-25:]))
        metrics = dict(run=name, failed=True, rc=rc, quotable=False, **cfg)
    with open(RESULTS, "a") as f:
        f.write(json.dumps(metrics) + "\n")
    return metrics

def results():
    import pandas as pd
    rows = {}
    for line in RESULTS.read_text().splitlines() if RESULTS.exists() else []:
        r = json.loads(line); rows[r["run"]] = r          # last run of a name wins
    return pd.DataFrame(rows.values())

def show(pattern, cols):
    df = results()
    df = df[df.run.str.match(pattern)]
    return df[[c for c in ["run", *cols] if c in df]]

def best(prefix, **match):
    df = results()
    df = df[df.run.str.startswith(prefix) & (df.quotable == True)]
    for k, v in match.items():
        df = df[df[k] == v]
    return df.sort_values("tok_s", ascending=False).iloc[0].to_dict() if len(df) else None

HF_TOKEN length: 37


### Device check (in a subprocess)

In [ ]:
!PYTHONPATH=/content/torchtpu_project python3 -c "import importlib.metadata as m, torch, torch_tpu, jax; \
print('torch_tpu', m.version('torch_tpu'), '| torch', torch.__version__, '| jax', jax.__version__); \
x = torch.ones(4, device='tpu'); print('tpu add ->', (x + 1).cpu().tolist()); \
print(torch.tpu.get_device_properties(0))"

I0000 00:00:1793459188.353736    2297 pjrt_api.cc:119] GetPjrtApi was found for TPU at /usr/local/lib/python3.13/dist-packages/libtpu/libtpu.so
I0000 00:00:1793459188.367398    2297 tpu_profiler_plugin.cc:800] PyTorch PrivateUse1ProfilerRegistry not found. Native TPU profiling will be disabled (this is expected on PyTorch < 2.12).
I0000 00:00:1793459189.994744    2297 device_rt.cc:166] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1793459190.038480    2297 libtpu_version.cc:63] Setting libtpu version to: 0.0.50
/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages 

## 1. CPU correctness, before any TPU run
* **Model:** the cached prefill + decode logits must equal an uncached full forward; fused and unfused layouts must give identical greedy tokens; in-place and functional KV must give identical greedy tokens.
* **Kernel:** the Pallas interpreter is compared against a float64 reference over padded, short, harness-max, and full windows for each `bkv`. Note that results are *not* bit-identical across block sizes (the online-softmax partition changes the rounding by about 1e-7 in f32). The draft's "identical for all block sizes" claim should be restated in those terms.

`TORCH_DEVICE_BACKEND_AUTOLOAD=0` and `JAX_PLATFORMS=cpu` keep both checks off the TPU.

In [ ]:
!cd /content/torchtpu_project && TORCH_DEVICE_BACKEND_AUTOLOAD=0 JAX_PLATFORMS=cpu python3 verify_model_cpu.py
!cd /content/torchtpu_project && TORCH_DEVICE_BACKEND_AUTOLOAD=0 JAX_PLATFORMS=cpu python3 verify_kernel_cpu.py

[MODEL VERIFY] loader consumed 29 tensors, fused layout
[MODEL VERIFY] cached decode vs uncached forward: max|dlogit| = 7.73e-06
[MODEL VERIFY] fused == unfused, in-place == functional (greedy tokens identical)
[MODEL VERIFY] PASS
/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
float32   pos= 15 max|err| per bkv: 64:1.1e-06 128:1.1e-06 256:1.1e-06  | spread across bkv: 0.0e+00
float32   pos= 40 max|err| per bkv: 64:9.3e-07 128:9.3e-07 256:9.3e-07  | spread across bkv: 0.0e+00
float32   pos=143 max|err| per bkv: 64:1.2e-06 128:1.0e-06 256:1.0e-06  | spread across bkv: 2.7e-07
float32   pos=255 max|err| per bkv: 64:7.5e-07 128:6.5e-07 256:6.5e-07  | spre

## Experiment 1: regime check at batch 128 (2 runs)
Eager JIT vs torch.compile, both in strict mode (the 0.2.0 default). On wheel 0.1.1 the compiled path paid a fixed ~430 ms per call; this shows whether 0.2.0 still does. The faster arm becomes `BEST` for the other experiments.

Evidence to check in the output: `eager_mode=...DEFER_NEVER`, `cpu_fallback False`, `grad_enabled_in_run False`, and an equal compiled-graph count after warmup and after main (zero recompiles while measuring).

In [ ]:
for comp in (0, 1):
    run(f"exp1_c{comp}_b128", BATCH_MULT=32, COMPILE=comp)
r = best("exp1_")
BEST = dict(COMPILE=int(r["compile"])) if r else dict(COMPILE=1)
print("BEST:", BEST)
show("exp1_", ["compile", "tok_s", "itl_ms", "ttft_ms", "warmup_s", "recompiled", "failed"])


>>> exp1_c0_b128  {'BATCH_MULT': 32, 'COMPILE': 0}
================ METRICS ================
Run:                           exp1_c0_b128
Regime:                        eager JIT | eager_mode=strict | attn=xla | sync_every=8 | inplace_kv=1 | fused_qkv=1 | decode_scan=0
Quotable:                      YES
[EVIDENCE] eager_mode               EagerMode.DEFER_NEVER
[EVIDENCE] cpu_fallback             False
[EVIDENCE] device                   TPU v5e | HBM 16.00 GiB
[EVIDENCE] tier2_cache              default
[EVIDENCE] tier3_cache              /content/torchtpu_cache
[EVIDENCE] weights                  random (valid throughput, invalid text)
[EVIDENCE] grad_enabled_in_run      False
[EVIDENCE] any_param_requires_grad  False
[EVIDENCE] compiled graphs          after warmup=0 after main=0 (equal => zero recompiles while measuring)
Model build + weights:             13.8 s
Warmup generate (compile):         2.54 s
Engine-init equivalent:            16.3 s  [build + warmup]
Batch TTFT (prefill+

,run,compile,tok_s,itl_ms,ttft_ms,warmup_s,recompiled
0,exp1_c0_b128,False,2095.0,60.73,107.7,2.54,False
1,exp1_c1_b128,True,10624.0,11.71,54.4,17.06,False


## Experiment 2: the kernel hack at batch 512 (4 runs)
The TorchTPU counterpart of the vLLM `RPA_BKV=256` override, which took 15,338 to 19,511 tok/s (+27%). It compares XLA attention against the Pallas decode kernel at `RPA_BKV` = 64, 128 and 256.

Evidence: `[MY PALLAS KERNEL] traced ...` fires only when the kernel is compiled onto the hot path, and `pallas_vs_xla_max_abs` is an on-TPU numerical check against the XLA attention. With `MAX_LEN=256` and at most 143 live positions, KV blocks entirely past `pos` are skipped (no DMA, no compute), so a smaller `bkv` skips more but runs more grid steps.

In [ ]:
run("exp2_xla_b512", BATCH_MULT=128, ATTN="xla", **BEST)
for bkv in (64, 128, 256):
    run(f"exp2_pallas_bkv{bkv}_b512", BATCH_MULT=128, ATTN="pallas", RPA_BKV=bkv, **BEST)
kb = best("exp2_")
ATTN_BEST = (dict(ATTN="pallas", RPA_BKV=int(kb["bkv"])) if kb and kb["attn"] == "pallas"
             else dict(ATTN="xla"))
print("ATTN_BEST:", ATTN_BEST)
show("exp2_", ["attn", "bkv", "tok_s", "itl_ms", "pallas_vs_xla_max_abs", "tokens_sha1", "failed"])


>>> exp2_xla_b512  {'BATCH_MULT': 128, 'ATTN': 'xla', 'COMPILE': 1}
================ METRICS ================
Run:                           exp2_xla_b512
Regime:                        torch.compile | eager_mode=strict | attn=xla | sync_every=8 | inplace_kv=1 | fused_qkv=1 | decode_scan=0
Quotable:                      YES
[EVIDENCE] eager_mode               EagerMode.DEFER_NEVER
[EVIDENCE] cpu_fallback             False
[EVIDENCE] device                   TPU v5e | HBM 16.00 GiB
[EVIDENCE] tier2_cache              default
[EVIDENCE] tier3_cache              /content/torchtpu_cache
[EVIDENCE] weights                  random (valid throughput, invalid text)
[EVIDENCE] grad_enabled_in_run      False
[EVIDENCE] any_param_requires_grad  False
[EVIDENCE] compiled graphs          after warmup=2 after main=2 (equal => zero recompiles while measuring)
Model build + weights:              2.9 s
Warmup generate (compile):        19.85 s
Engine-init equivalent:            22.8 s  [build + warmup

,run,attn,bkv,tok_s,itl_ms,pallas_vs_xla_max_abs,tokens_sha1
2,exp2_xla_b512,xla,NaN,16923.2,28.42,NaN,ef58b9f6c2b1
3,exp2_pallas_bkv64_b512,pallas,64.0,17283.4,28.07,0.011719,ef58b9f6c2b1
4,exp2_pallas_bkv128_b512,pallas,128.0,17620.4,27.50,0.011719,ef58b9f6c2b1
5,exp2_pallas_bkv256_b512,pallas,256.0,17872.0,27.07,0.011719,ef58b9f6c2b1


## Experiment 3: multi-step decode with `scan` at batch 512 (2 runs, experimental)
The CUDA Graphs analogue from draft §8.1. `DECODE_SCAN=K` compiles K greedy decode steps into one `stablehlo.while` executable, so the host crosses the compiled boundary 127/K times instead of 127. The KV caches are carried through the loop. If lowering fails, the run is recorded as a failure (a ledger entry for the maintainers) and the notebook continues.

In [ ]:
for k in (8, 32):
    run(f"exp3_scan{k}_b512", BATCH_MULT=128, DECODE_SCAN=k, **BEST, **ATTN_BEST)
show("exp[23]_", ["decode_scan", "attn", "tok_s", "itl_ms", "warmup_s", "tokens_sha1", "failed"])


>>> exp3_scan8_b512  {'BATCH_MULT': 128, 'DECODE_SCAN': 8, 'COMPILE': 1, 'ATTN': 'pallas', 'RPA_BKV': 256}
[MY PALLAS KERNEL] traced q=(512, 8, 256) kv=(512, 256, 256) bkv=256 bb=8
================ METRICS ================
Run:                           exp3_scan8_b512
Regime:                        torch.compile | eager_mode=strict | attn=pallas bkv=256 bb=8 | sync_every=8 | inplace_kv=1 | fused_qkv=1 | decode_scan=8
Quotable:                      YES
[EVIDENCE] eager_mode               EagerMode.DEFER_NEVER
[EVIDENCE] cpu_fallback             False
[EVIDENCE] device                   TPU v5e | HBM 16.00 GiB
[EVIDENCE] tier2_cache              default
[EVIDENCE] tier3_cache              /content/torchtpu_cache
[EVIDENCE] weights                  random (valid throughput, invalid text)
[EVIDENCE] grad_enabled_in_run      False
[EVIDENCE] any_param_requires_grad  False
[EVIDENCE] pallas_vs_xla_max_abs    0.01171875
[EVIDENCE] compiled graphs          after warmup=3 after main=3 (equal 

,run,decode_scan,attn,tok_s,itl_ms,warmup_s,tokens_sha1
2,exp2_xla_b512,0,xla,16923.2,28.42,19.85,ef58b9f6c2b1
3,exp2_pallas_bkv64_b512,0,pallas,17283.4,28.07,10.55,ef58b9f6c2b1
4,exp2_pallas_bkv128_b512,0,pallas,17620.4,27.50,10.61,ef58b9f6c2b1
5,exp2_pallas_bkv256_b512,0,pallas,17872.0,27.07,10.68,ef58b9f6c2b1
6,exp3_scan8_b512,8,pallas,21639.0,22.10,19.75,ef58b9f6c2b1
7,exp3_scan32_b512,32,pallas,21320.5,22.45,17.22,ef58b9f6c2b1


## Experiment 4: real weights at batch 512, the headline number (1 run)
`google/gemma-2b` with the best configuration from experiments 1 to 3. This is the number that sits next to vLLM's 19,511 (the vLLM study used real weights). `DETERMINISM=1` runs the main generation twice and requires bit-identical tokens; sample text is printed for 4 prompts.

Requires access to `google/gemma-2b` with `HF_TOKEN` in Colab secrets.

In [ ]:
s3 = best("exp3_")
SCAN_BEST = (dict(DECODE_SCAN=int(s3["decode_scan"]))
             if s3 and kb and s3["tok_s"] > kb["tok_s"] else {})
print("final config:", {**BEST, **ATTN_BEST, **SCAN_BEST})
run("exp4_real_b512", BATCH_MULT=128, MODEL="google/gemma-2b", DETERMINISM=1,
    **BEST, **ATTN_BEST, **SCAN_BEST)

final config: {'COMPILE': 1, 'ATTN': 'pallas', 'RPA_BKV': 256, 'DECODE_SCAN': 8}

>>> exp4_real_b512  {'BATCH_MULT': 128, 'MODEL': 'google/gemma-2b', 'DETERMINISM': 1, 'COMPILE': 1, 'ATTN': 'pallas', 'RPA_BKV': 256, 'DECODE_SCAN': 8}
[MY PALLAS KERNEL] traced q=(512, 8, 256) kv=(512, 256, 256) bkv=256 bb=8
================ METRICS ================
Run:                           exp4_real_b512
Regime:                        torch.compile | eager_mode=strict | attn=pallas bkv=256 bb=8 | sync_every=8 | inplace_kv=1 | fused_qkv=1 | decode_scan=8
Quotable:                      YES
[EVIDENCE] eager_mode               EagerMode.DEFER_NEVER
[EVIDENCE] cpu_fallback             False
[EVIDENCE] device                   TPU v5e | HBM 16.00 GiB
[EVIDENCE] tier2_cache              default
[EVIDENCE] tier3_cache              /content/torchtpu_cache
[EVIDENCE] weights                  google/gemma-2b (164 tensors, all consumed)
[EVIDENCE] grad_enabled_in_run      False
[EVIDENCE] any_param_requires_g

{'run': 'exp4_real_b512',
 'quotable': True,
 'batch': 512,
 'model': 'google/gemma-2b',
 'compile': True,
 'exec_mode': 'strict',
 'attn': 'pallas',
 'bkv': 256,
 'bb': 8,
 'sync_every': 8,
 'inplace_kv': True,
 'fused_qkv': True,
 'decode_scan': 8,
 'tok_s': 21645.8,
 'itl_ms': 22.09,
 'ttft_ms': 221.9,
 'build_s': 45.9,
 'warmup_s': 11.4,
 'graphs': 3,
 'recompiled': False,
 'step_median_ms': None,
 'determinism': True,
 'tokens_sha1': 'a49fbe084f60',
 'mj_per_token': 9.1,
 'pallas_vs_xla_max_abs': 0.01171875,
 'peak_hbm_gib': 15.685115814208984}

## Results and comparison with the vLLM / JAX study
Only quotable runs appear in the table; failed runs are listed separately. The reference numbers come from the vLLM notebook output (batch 512, BKV=256) and the draft.

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
df = results()
q = df[df.quotable == True].copy()
cols = ["run", "batch", "model", "compile", "attn", "bkv", "decode_scan", "tok_s", "itl_ms",
        "ttft_ms", "warmup_s", "mj_per_token", "recompiled", "determinism"]
display(q[[c for c in cols if c in q]])
fails = df[df.get("failed", pd.Series(False, index=df.index)) == True]
if len(fails):
    print("FAILED runs (ledger entries):"); display(fails[["run", "rc"]])

VLLM_BEST, VLLM_STOCK, DRAFT_EAGER_512 = 19511.2, 15338, 5158
h = df[df.run == "exp4_real_b512"]
if len(h) and pd.notna(h.iloc[0].get("tok_s")):
    h = h.iloc[0]
    print(f"TorchTPU 0.2.0, real weights, B=512: {h.tok_s:,.0f} tok/s | TTFT {h.ttft_ms:.0f} ms "
          f"| ITL {h.itl_ms:.1f} ms | determinism {h.determinism}")
    print(f"  vs vLLM best (BKV=256) {VLLM_BEST:,.0f}: x{h.tok_s / VLLM_BEST:.2f}")
    print(f"  vs vLLM stock RPA      {VLLM_STOCK:,}: x{h.tok_s / VLLM_STOCK:.2f}")
    print(f"  vs draft 0.1.1 eager   {DRAFT_EAGER_512:,}: x{h.tok_s / DRAFT_EAGER_512:.2f}")

,run,batch,model,compile,attn,bkv,decode_scan,tok_s,itl_ms,ttft_ms,warmup_s,mj_per_token,recompiled,determinism
0,exp1_c0_b128,128,random,False,xla,NaN,0,2095.0,60.73,107.7,2.54,94.0,False,None
1,exp1_c1_b128,128,random,True,xla,NaN,0,10624.0,11.71,54.4,17.06,18.5,False,None
2,exp2_xla_b512,512,random,True,xla,NaN,0,16923.2,28.42,263.1,19.85,11.6,False,None
3,exp2_pallas_bkv64_b512,512,random,True,pallas,64.0,0,17283.4,28.07,227.5,10.55,11.4,False,None
4,exp2_pallas_bkv128_b512,512,random,True,pallas,128.0,0,17620.4,27.50,227.0,10.61,11.2,False,None
5,exp2_pallas_bkv256_b512,512,random,True,pallas,256.0,0,17872.0,27.07,228.9,10.68,11.0,False,None
6,exp3_scan8_b512,512,random,True,pallas,256.0,8,21639.0,22.10,222.2,19.75,9.1,False,None
7,exp3_scan32_b512,512,random,True,pallas,256.0,32,21320.5,22.45,222.2,17.22,9.2,False,None
8,exp4_real_b512,512,google/gemma-2b,True,pallas,256.0,8,21645.8,22.09,221.9,11.40,9.1,False,True


TorchTPU 0.2.0, real weights, B=512: 21,646 tok/s | TTFT 222 ms | ITL 22.1 ms | determinism True
  vs vLLM best (BKV=256) 19,511: x1.11
  vs vLLM stock RPA      15,338: x1.41
  vs draft 0.1.1 eager   5,158: x4.20


### What these results update in the draft
* **Tables 3 to 5 / §7.2 to 7.4:** experiment 1 says whether the fixed compiled-call cost survives on 0.2.0 (strict is now the default).
* **§7.5:** experiment 2 replaces the masked kernel sweep, including block skipping and the on-TPU `pallas_vs_xla_max_abs`.
* **§8.1:** experiment 3 is the `scan` result (a speedup, or a failure with its error).
* **§8.2:** experiment 4 is the new headline against 19,511.
* Independent of the runs: correct "24 percent" to 27% in §3.4, and update the §5 software line to torch_tpu 0.2.0, torch 2.11.0+cpu, jax 0.9.2, libtpu 0.0.50.